[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/manojkumar12-maker/AIVG-MAX-D/blob/main/notebooks/OpenGenerativeAI_Colab.ipynb)
# Open Generative AI — Self-Hosted Colab GPU backend

Runs **ComfyUI** plus the **FastAPI adapter** on a free Google Colab GPU and exposes them to the
static frontend (GitHub Pages) through a public **cloudflared** tunnel.

Run the sections in order: **A → B → C → D → E**. Section E prints a
`Frontend URL` and a `Session token` — paste both into the frontend Connection panel.

Notes
- Colab runtimes are temporary. When the runtime resets, reconnect and re-run A→E (section C caches models to Google Drive if you enabled it in A).
- Tunnels are for personal use, not production.
- Models are open weights; check each licence shown in section C before commercial use.

## A — Environment check and options
Confirms a GPU is attached and sets the options used by later sections.
Set `MOUNT_DRIVE = True` to store models and outputs on your Google Drive so they survive runtime resets.

In [ ]:
#@title A — Environment check {display-mode:"form"}
MOUNT_DRIVE = True   #@param {type:"boolean"}
COMFYUI_DIR = "/content/ComfyUI"
REPO_URL    = "https://github.com/Anil-matcha/Open-Generative-AI.git"  # or your fork
REPO_DIR    = "/content/Open-Generative-AI"

import torch, shutil, subprocess
assert torch.cuda.is_available(), "No GPU: use Runtime > Change runtime type > T4 GPU"
name  = torch.cuda.get_device_name(0)
vram  = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {name} ({vram:.1f} GB VRAM)")
if vram < 20:
    print("Low-VRAM GPU: stick to SD 1.5 and SDXL workflows; video models will likely not fit.")

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    os.makedirs('/content/drive/MyDrive/OpenGenerativeAI/models', exist_ok=True)
    os.makedirs('/content/drive/MyDrive/OpenGenerativeAI/outputs', exist_ok=True)
    print("Drive storage: /content/drive/MyDrive/OpenGenerativeAI")

## B — Install ComfyUI and the adapter

In [ ]:
#@title B — Install {display-mode:"form"}
import os
if not os.path.exists(COMFYUI_DIR):
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI {COMFYUI_DIR}
!pip -q install -r {COMFYUI_DIR}/requirements.txt
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
!pip -q install -r {REPO_DIR}/backend/requirements.txt
print("ComfyUI + adapter installed.")

## C — Choose models
Pick a preset. Each preset lists what is downloaded, its disk size and its licence.
SD 1.5 fits any GPU; SDXL wants ~12 GB; FLUX schnell Q4 wants ~16 GB; Wan 2.1 video wants ~24 GB.

In [ ]:
#@title C — Model preset {display-mode:"form"}
PRESET = "SD 1.5 (fastest, any GPU)"  #@param ["SD 1.5 (fastest, any GPU)", "SDXL (high quality, ~12GB)", "SDXL + FLUX schnell Q4 (~16GB)", "Everything incl. Wan 2.1 video (~24GB)"]

PRESETS = {
    "SD 1.5 (fastest, any GPU)": [
        ("checkpoints/",
         "https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly-fp16.safetensors",
         "sd_v1-5_pruned_emaonly_fp16.safetensors",
         "~2 GB — CreativeML OpenRAIL-M (non-commercial restrictions may apply)")],
    "SDXL (high quality, ~12GB)": [
        ("checkpoints/",
         "https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors",
         "sd_xl_base_1.0.safetensors",
         "~6.9 GB — CreativeML Open RAIL++-M")],
    "SDXL + FLUX schnell Q4 (~16GB)": [
        ("checkpoints/",
         "https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors",
         "sd_xl_base_1.0.safetensors",
         "~6.9 GB — CreativeML Open RAIL++-M"),
        ("unet/",
         "https://huggingface.co/city96/FLUX.1-schnell-gguf/resolve/main/flux1-schnell-Q4_K_S.gguf",
         "flux1-schnell-Q4_K_S.gguf",
         "~5.9 GB — FLUX.1 [schnell] non-commercial license"),
        ("clip/",
         "https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp8_e4m3fn.safetensors",
         "t5xxl_fp8_e4m3fn.safetensors",
         "~4.9 GB — Apache-2.0 (T5)"),
        ("vae/",
         "https://huggingface.co/black-forest-labs/FLUX.1-schnell/resolve/main/ae.safetensors",
         "ae_schnell.safetensors",
         "~0.3 GB — FLUX.1 [schnell] non-commercial license")],
    "Everything incl. Wan 2.1 video (~24GB)": [
        ("checkpoints/",
         "https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors",
         "sd_xl_base_1.0.safetensors",
         "~6.9 GB — CreativeML Open RAIL++-M"),
        ("diffusion_models/",
         "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/diffusion_models/wan2.1_t2v_1.3B_fp16.safetensors",
         "wan2.1_t2v_1.3B_fp16.safetensors",
         "~2.6 GB — Apache-2.0"),
        ("text_encoders/",
         "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors",
         "umt5_xxl_fp8_e4m3fn_scaled.safetensors",
         "~6.4 GB — Apache-2.0"),
        ("vae/",
         "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/vae/wan_2.1_vae.safetensors",
         "wan_2.1_vae.safetensors",
         "~0.25 GB — Apache-2.0")],
}
for _, url, fname, lic in PRESETS[PRESET]:
    print(f"- {fname}: {lic}")

## D — Download models (resumable, cached to Drive if mounted)

In [ ]:
#@title D — Download {display-mode:"form"}
import os, subprocess, shutil
DRIVE = '/content/drive/MyDrive/OpenGenerativeAI/models' if MOUNT_DRIVE else None

def fetch(subdir, url, fname):
    dst = os.path.join(COMFYUI_DIR, 'models', subdir, fname)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    cache = os.path.join(DRIVE, subdir, fname) if DRIVE else None
    if cache and os.path.exists(cache) and not os.path.exists(dst):
        print(f"Linking {fname} from Drive cache")
        os.symlink(cache, dst)
        return
    if os.path.exists(dst):
        print(f"Already have {fname}")
        return
    print(f"Downloading {fname} ...")
    tmp = dst + '.part'
    rc = subprocess.run(['wget', '-c', '-q', '--show-progress', '-O', tmp, url]).returncode
    assert rc == 0, f"Download failed: {url}"
    os.replace(tmp, dst)
    if cache:
        os.makedirs(os.path.dirname(cache), exist_ok=True)
        shutil.copy2(dst, cache)

for item in PRESETS[PRESET]:
    fetch(*item)
print("Model downloads complete.")

## E — Start ComfyUI + adapter, open the tunnel
Prints the **Frontend URL** and **Session token** to paste into the Connection panel.
Keep this cell running; stop it with the stop button when you are done.

In [ ]:
#@title E — Run services and print connection details {display-mode:"form"}
import os, secrets, subprocess, time

# Self-sufficient: derive VRAM/paths here so E works even if A was skipped
# or the runtime restarted and cell A variables are gone.
try:
    COMFYUI_DIR, REPO_DIR, MOUNT_DRIVE
except NameError:
    MOUNT_DRIVE = os.path.isdir('/content/drive/MyDrive')
    REPO_DIR = '/content/OpenGenerativeAI'
    COMFYUI_DIR = os.path.join(REPO_DIR, 'ComfyUI')
vram = 0
try:
    import torch
    if torch.cuda.is_available():
        vram = torch.cuda.get_device_properties(0).total_memory / 2**30
except Exception:
    pass
print(f'Detected GPU VRAM: {vram:.1f} GB' if vram else 'No GPU detected')

TOKEN = secrets.token_urlsafe(24)
OUTDIR = '/content/drive/MyDrive/OpenGenerativeAI/outputs' if MOUNT_DRIVE else '/content/outputs'
os.makedirs(OUTDIR, exist_ok=True)

# Start ComfyUI (local API only)
comfy = subprocess.Popen(
    ['python', 'main.py', '--listen', '127.0.0.1', '--port', '8188',
     *([] if vram >= 20 else (['--lowvram'] if vram >= 12 else ['--lowvram', '--novram']))],
    cwd=COMFYUI_DIR, stdout=open('/content/comfyui.log', 'w'), stderr=subprocess.STDOUT)

# Start the adapter
env = dict(os.environ,
           OGAI_SESSION_TOKEN=TOKEN,
           OGAI_COMFY_URL='http://127.0.0.1:8188',
           OGAI_OUTPUT_DIR=OUTDIR,
           **({'OGAI_DRIVE_DIR': '/content/drive/MyDrive/OpenGenerativeAI'} if MOUNT_DRIVE else {}),
           OGAI_STATE_DIR='/content/state')
adapter = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'backend.adapter.main:app', '--host', '127.0.0.1', '--port', '8189'],
    cwd=REPO_DIR, env=env, stdout=open('/content/adapter.log', 'w'), stderr=subprocess.STDOUT)

# Tunnel
!wget -q -O /usr/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/bin/cloudflared
tunnel = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8189', '--no-autoupdate'],
                          stdout=open('/content/tunnel.log', 'w'), stderr=subprocess.STDOUT)

url = None
for _ in range(60):
    time.sleep(2)
    try:
        text = open('/content/tunnel.log').read()
        for line in text.splitlines():
            if 'trycloudflare.com' in line:
                url = line.split('trycloudflare.com')[0].rsplit('http', 1)[-1].strip() or None
                import re
                m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', text)
                if m: url = m.group(0)
                break
    except FileNotFoundError:
        pass
    if url: break

print('=' * 60)
print(f'Frontend URL : {url or "(not ready - rerun this cell)"}')
print(f'Session token: {TOKEN}')
print('=' * 60)
print('Save to Google Drive:', 'ON (toggle it in Settings)' if MOUNT_DRIVE else 'off (mount Drive in cell A)')
print('Paste both into the frontend Connection panel.')
print('Logs: /content/comfyui.log /content/adapter.log /content/tunnel.log')